In [3]:
import numpy as np
import pandas as pd

from MLstatkit import Delong_test
from model_evaluation import leave_one_patient_out_logistic_regression as reg, delta_model
from ar_model_utils import apply_sliding_window
from data_utils import *
from plot_utils import *
from model_evaluation import *

from tqdm import tqdm
from joblib import Parallel, delayed
import os 
import warnings
warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

In [4]:
df = pd.read_parquet('data/df_for_paper_processed.parq')
scale_df = pd.read_parquet('data/scale_df_for_paper.parq')
df.head(5)

,pt_id,time_bin_time,days_since_dbs,lfp_left_raw,stim_left,lfp_right_raw,stim_right,lead_location,left_lead_model,right_lead_model,...,left_R2_rolling_avg_14d,left_R2_rolling_avg_21d,left_R2_rolling_avg_28d,left_delta_R2_rolling_avg_1d,left_delta_R2_rolling_avg_3d,left_delta_R2_rolling_avg_5d,left_delta_R2_rolling_avg_7d,left_delta_R2_rolling_avg_14d,left_delta_R2_rolling_avg_21d,left_delta_R2_rolling_avg_28d
0,AA001,22:30:00,-8,538.0,0.0,651.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AA001,22:40:00,-8,332.0,0.0,432.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AA001,22:50:00,-8,715.0,0.0,873.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AA001,23:00:00,-8,301.0,0.0,392.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AA001,23:10:00,-8,374.0,0.0,533.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Compile right hem features
window_size = 3

df.reset_index(drop=True, inplace=True)
hem = 'right'

pt_groups = df.groupby('pt_id', group_keys=False)
ar_features = [f'lfp_{hem}_OvER_interpolate_z_scored_lag_1']
tqdm.pandas(desc=f'Applying autoregressive model for {hem} hem')
hem_results_df = pt_groups.progress_apply(
    lambda g: apply_sliding_window(
        g, ar_features, f'lfp_{hem}_OvER_interpolate_z_scored', window_size=window_size),
    include_groups=False
)
df = df.merge(
    hem_results_df,
    how='left',
    left_index=True,
    right_index=True
)

Applying autoregressive model for right hem: 100%|██████████| 24/24 [00:13<00:00,  1.73it/s]


# MNI Coordinates Table

MNI coordinates were obtained via preoperative CT and MRI. See methods for details.

# Patient Demographics, Clinical Outcomes, and Recording Durations

In [3]:
gender_dict = {
    'AA001': 'F',
    'AA002': 'M',
    'AA004': 'M',
    'B001': 'F',
    'B002': 'F',
    'B004': 'M',
    'B005': 'M',
    'B006': 'F',
    'B007': 'M',
    'B008': 'F',
    'B009': 'F',
    'B010': 'F',
    'B011': 'M',
    'B012': 'F',
    'B013': 'M',
    'B014': 'M',
    'B015': 'F',
    'B017': 'M',
    'B019': 'M',
    'B020': 'M',
    'U001': 'F',
    'U002': 'M',
    'U003': 'F',
    'U004': 'F'
}

age_dict = {
    'AA001': 19,
    'AA002': 26,
    'AA004': 58,
    'B001': 42,
    'B002': 31,
    'B004': 31,
    'B005': 20,
    'B006': 55,
    'B007': 23,
    'B008': 33,
    'B009': 31,
    'B010': 21,
    'B011': 46,
    'B012': 38,
    'B013': 23,
    'B014': 36,
    'B015': 27,
    'B017': 33,
    'B019': 37,
    'B020': 40,
    'U001': 33,
    'U002': 23,
    'U003': 24,
    'U004': 44
}

followup_dur_dict = {
    'AA001': 10,
    'AA002': 12,
    'AA004': 6,
    'B001': 24,
    'B002': 2,
    'B004': 38,
    'B005': 34,
    'B006': 43,
    'B007': 25,
    'B008': 37,
    'B009': 53,
    'B010': 12,
    'B011': 13,
    'B012': 59,
    'B013': 11,
    'B014': 45,
    'B015': 32,
    'B017': 38,
    'B019': 18,
    'B020': 65,
    'U001': 2,
    'U002': 4,
    'U003': 10,
    'U004': 1
}

responder_status_dict = {}
for pt_id, pt_scale_df in scale_df.groupby('pt_id'):
    if pt_id in ['AA001', 'AA002', 'AA004']:
        response_status = 'Unknown'
    elif pt_scale_df['is_clinical_response'].any():
        response_status = 'Yes'
    elif pt_scale_df['YBOCS_reduction'].max() > 0.25:
        response_status = 'Partial'
    else:
        response_status = 'No'
    responder_status_dict[pt_id] = response_status

baseline_ybocs_dict, final_ybocs_dict = {}, {}
for pt_id, pt_scale_df in scale_df.groupby('pt_id'):
    baseline_ybocs_dict[pt_id] = pt_scale_df.query('YBOCS_reduction == 0')['YBOCS'].iloc[0]
    final_ybocs_dict[pt_id] = pt_scale_df['YBOCS'].iloc[-1]

left_recording_dur_dict, right_recording_dur_dict = {}, {}
for pt_id, pt_df in df.groupby('pt_id', sort=False):
    for hem in ['left', 'right']:
        hem_df = pt_df.dropna(subset=[f'lfp_{hem}_OvER_interpolate_z_scored'])
        unique_days = hem_df['days_since_dbs'].nunique()
        if hem == 'left':
            left_recording_dur_dict[pt_id] = unique_days
        else:
            right_recording_dur_dict[pt_id] = unique_days

pt_demo_df = pd.DataFrame({
    'pt_id': list(gender_dict.keys()),
    'gender': list(gender_dict.values()),
    'age': list(age_dict.values()),
    'followup_dur': list(followup_dur_dict.values()),
    'responder_status': [responder_status_dict[pt_id] for pt_id in gender_dict.keys()],
    'baseline_ybocs': [baseline_ybocs_dict[pt_id] for pt_id in gender_dict.keys()],
    'final_ybocs': [final_ybocs_dict[pt_id] for pt_id in gender_dict.keys()],
    'left_recording_dur': [left_recording_dur_dict[pt_id] for pt_id in gender_dict.keys()],
    'right_recording_dur': [right_recording_dur_dict[pt_id] for pt_id in gender_dict.keys()],
})
pt_demo_df.to_excel('tables/pt_demo.xlsx', index=False)
pt_demo_df

,pt_id,gender,age,followup_dur,responder_status,baseline_ybocs,final_ybocs,left_recording_dur,right_recording_dur
0,AA001,F,19,10,Unknown,37.0,32.0,141,27
1,AA002,M,26,12,Unknown,37.0,35.0,136,65
2,AA004,M,58,6,Unknown,36.0,26.0,95,94
3,B001,F,42,24,Yes,38.0,24.0,650,650
4,B002,F,31,2,No,40.0,39.0,38,50
5,B004,M,31,38,Yes,34.0,11.0,544,11
6,B005,M,20,34,Yes,34.0,10.0,482,482
7,B006,F,55,43,Partial,39.0,37.0,829,917
8,B007,M,23,25,Yes,33.0,28.0,15,97
9,B008,F,33,37,Partial,35.0,26.0,382,361


# Pre- vs post-DBS stats with and without sample size correction

In [4]:
_NAN_STATS = {
    't_stat': np.nan, 'p_val': np.nan, 'dof': np.nan, 'ci': (np.nan, np.nan),
    'hedges_g': np.nan, 'hedges_g_ci': (np.nan, np.nan),
    'n1': np.nan, 'n2': np.nan,
}

feature = 'lfp_left_OvER_interpolate_R2'

results_dict = {'pt_id': [], 'response_status': []}
for corr in ['raw', 'neff']:
    for m in ['n1', 'n2', 'dof', 't_stat', 'p_val', 'ci', 'hedges_g', 'hedges_g_ci']:
        results_dict[f'{m}_{corr}'] = []

max_num_pts = 0
df.sort_values(by=['pt_id', 'dummy_timestamp'], inplace=True)
for response_status in ['Responder', 'Non-Responder', 'Unknown']:
    table_df = df.dropna(subset=feature).query('response_status == @response_status').copy()

    max_num_pts = max(max_num_pts, table_df['pt_id'].nunique())
    for i, (pt_id, pt_df) in enumerate(table_df.groupby('pt_id')):
        table_df_daily = pt_df.dropna(subset=[feature]).groupby('days_since_dbs').first().reset_index()
        predbs_days, predbs_vals = table_df_daily.query('state_label_str == "Pre-DBS"')[['days_since_dbs', feature]].values.T
        stable_state_days, stable_state_vals = table_df_daily.query('state_label_str == @response_status')[['days_since_dbs', feature]].values.T

        # statistical tests
        if len(predbs_vals) > 1 and len(stable_state_vals) > 1 and not np.isnan(predbs_vals).all() and not np.isnan(stable_state_vals).all():
            n1_raw, n2_raw = len(predbs_vals), len(stable_state_vals)
            raw_stats = welch_stats_with_effect_size(predbs_vals, stable_state_vals, n1_raw, n2_raw)
            try:
                neff1 = compute_neff(predbs_days, predbs_vals)
                neff2 = compute_neff(stable_state_days, stable_state_vals)
                if neff1 > 1 and neff2 > 1:
                    neff_stats = welch_stats_with_effect_size(predbs_vals, stable_state_vals, neff1, neff2)
                else:
                    neff_stats = dict(_NAN_STATS)
            except Exception:
                neff_stats = dict(_NAN_STATS)

            t_stat, p_val = raw_stats['t_stat'], raw_stats['p_val']
        else:
            raw_stats = dict(_NAN_STATS)
            neff_stats = dict(_NAN_STATS)
            t_stat, p_val = np.nan, np.nan

        results_dict['pt_id'].append(pt_id)
        results_dict['response_status'].append(response_status)
        for corr, stat_dict in [('raw', raw_stats), ('neff', neff_stats)]:
            for m in ['n1', 'n2', 'dof', 't_stat', 'p_val', 'ci', 'hedges_g', 'hedges_g_ci']:
                results_dict[f'{m}_{corr}'].append(stat_dict[m])

results_df = pd.DataFrame(results_dict)
results_df.sort_values(by='pt_id', inplace=True)
results_df.to_excel('tables/pt_stats.xlsx', index=False)
results_df

,pt_id,response_status,n1_raw,n2_raw,dof_raw,t_stat_raw,p_val_raw,ci_raw,hedges_g_raw,hedges_g_ci_raw,n1_neff,n2_neff,dof_neff,t_stat_neff,p_val_neff,ci_neff,hedges_g_neff,hedges_g_ci_neff
20,AA001,Unknown,7.0,128.0,9.490733,6.392403,9.992490e-05,"(0.1728423394321971, 0.35989196702765514)",1.260011,"(0.48874105872995344, 2.031281657077019)",5.644774,13.943656,17.053155,3.768783,1.523985e-03,"(0.11728662746162366, 0.41544767899822854)",1.338205,"(0.31316484981856196, 2.3632446380968624)"
21,AA002,Unknown,15.0,114.0,24.620707,6.260447,1.607851e-06,"(0.15420209846751537, 0.30557662570978933)",1.204164,"(0.6492180920397341, 1.7591093247025962)",4.906006,29.673875,7.704768,3.470307,8.946888e-03,"(0.07610394697479506, 0.38367477720250964)",1.186968,"(0.21286317006825184, 2.1610720128981042)"
22,AA004,Unknown,8.0,85.0,16.561204,-4.125275,7.428514e-04,"(-0.18429649839091788, -0.059406088010501845)",-0.766414,"(-1.4936415212755185, -0.03918718083850126)",3.487272,24.198377,7.932052,-2.496512,3.738400e-02,"(-0.2345721931717603, -0.009130393229659417)",-0.756735,"(-1.8643516716342639, 0.3508813909373575)"
0,B001,Responder,47.0,540.0,72.759679,30.066253,8.983927e-43,"(0.3651447224725812, 0.4169930543137989)",2.916371,"(2.5751248395146593, 3.2576180647397104)",12.521504,66.185550,26.799966,13.918570,8.801022e-14,"(0.33339863948976295, 0.44873913729661713)",2.967172,"(2.210496356641882, 3.723848387396029)"
13,B002,Non-Responder,5.0,12.0,6.425885,0.045601,9.650125e-01,"(-0.26560145943216296, 0.2758546600134041)",0.025011,"(-0.9652476916360923, 1.015269372823066)",3.173602,6.828463,3.644412,0.035872,9.732615e-01,"(-0.4074052671837189, 0.41765846776496)",0.023777,"(-1.1789862090859764, 1.2265409025782887)"
1,B004,Responder,8.0,322.0,9.023150,14.970022,1.116522e-07,"(0.4259227427044547, 0.5774919433711793)",2.430141,"(1.7060995090757767, 3.154181697931229)",4.391549,54.489569,6.945629,9.850153,2.487941e-05,"(0.3810762685350041, 0.6223384175406298)",2.442637,"(1.3867030845539425, 3.4985706592366945)"
2,B005,Responder,43.0,385.0,55.897011,9.969841,5.204908e-14,"(0.20256015112866432, 0.3044348256028766)",1.403493,"(1.075157586531507, 1.7318288426981343)",8.164305,75.453630,9.459405,4.353371,1.636810e-03,"(0.12274048795674675, 0.3842544887747942)",1.390846,"(0.6449925166342585, 2.136699054557984)"
14,B006,Non-Responder,13.0,787.0,15.925697,5.193194,9.015394e-05,"(0.033198863822507904, 0.07902831451676152)",0.512273,"(-0.03585449395198037, 1.0604000351689993)",7.158043,56.579514,25.105735,2.810702,9.447063e-03,"(0.015005198711694097, 0.09722197962757534)",0.527074,"(-0.24640656580338904, 1.3005536517319696)"
3,B007,Responder,NaN,NaN,NaN,NaN,NaN,"(nan, nan)",NaN,"(nan, nan)",NaN,NaN,NaN,NaN,NaN,"(nan, nan)",NaN,"(nan, nan)"
15,B008,Non-Responder,18.0,351.0,21.079652,0.341695,7.359633e-01,"(-0.08101729139376973, 0.11288394461522427)",0.057659,"(-0.41505571940377, 0.5303739754339656)",3.882366,27.917833,4.887843,0.146395,8.894550e-01,"(-0.26578481393268777, 0.2976514671541423)",0.057129,"(-0.9776534644715286, 1.0919111773914159)"


In [7]:
# Right hemisphere
_NAN_STATS = {
    't_stat': np.nan, 'p_val': np.nan, 'dof': np.nan, 'ci': (np.nan, np.nan),
    'hedges_g': np.nan, 'hedges_g_ci': (np.nan, np.nan),
    'n1': np.nan, 'n2': np.nan,
}

feature = 'lfp_right_OvER_interpolate_R2'

results_dict = {'pt_id': [], 'response_status': []}
for corr in ['raw', 'neff']:
    for m in ['n1', 'n2', 'dof', 't_stat', 'p_val', 'ci', 'hedges_g', 'hedges_g_ci']:
        results_dict[f'{m}_{corr}'] = []

max_num_pts = 0
df.sort_values(by=['pt_id', 'dummy_timestamp'], inplace=True)
for response_status in ['Responder', 'Non-Responder', 'Unknown']:
    table_df = df.dropna(subset=feature).query('response_status == @response_status').copy()

    max_num_pts = max(max_num_pts, table_df['pt_id'].nunique())
    for i, (pt_id, pt_df) in enumerate(table_df.groupby('pt_id')):
        table_df_daily = pt_df.dropna(subset=[feature]).groupby('days_since_dbs').first().reset_index()
        predbs_days, predbs_vals = table_df_daily.query('state_label_str == "Pre-DBS"')[['days_since_dbs', feature]].values.T
        stable_state_days, stable_state_vals = table_df_daily.query('state_label_str == @response_status')[['days_since_dbs', feature]].values.T

        # statistical tests
        if len(predbs_vals) > 1 and len(stable_state_vals) > 1 and not np.isnan(predbs_vals).all() and not np.isnan(stable_state_vals).all():
            n1_raw, n2_raw = len(predbs_vals), len(stable_state_vals)
            raw_stats = welch_stats_with_effect_size(predbs_vals, stable_state_vals, n1_raw, n2_raw)
            try:
                neff1 = compute_neff(predbs_days, predbs_vals)
                neff2 = compute_neff(stable_state_days, stable_state_vals)
                if neff1 > 1 and neff2 > 1:
                    neff_stats = welch_stats_with_effect_size(predbs_vals, stable_state_vals, neff1, neff2)
                else:
                    neff_stats = dict(_NAN_STATS)
            except Exception:
                neff_stats = dict(_NAN_STATS)

            t_stat, p_val = raw_stats['t_stat'], raw_stats['p_val']
        else:
            raw_stats = dict(_NAN_STATS)
            neff_stats = dict(_NAN_STATS)
            t_stat, p_val = np.nan, np.nan

        results_dict['pt_id'].append(pt_id)
        results_dict['response_status'].append(response_status)
        for corr, stat_dict in [('raw', raw_stats), ('neff', neff_stats)]:
            for m in ['n1', 'n2', 'dof', 't_stat', 'p_val', 'ci', 'hedges_g', 'hedges_g_ci']:
                results_dict[f'{m}_{corr}'].append(stat_dict[m])

results_df = pd.DataFrame(results_dict)
results_df.sort_values(by='pt_id', inplace=True)
results_df.to_excel('tables/right_pt_stats.xlsx', index=False)
results_df

,pt_id,response_status,n1_raw,n2_raw,dof_raw,t_stat_raw,p_val_raw,ci_raw,hedges_g_raw,hedges_g_ci_raw,n1_neff,n2_neff,dof_neff,t_stat_neff,p_val_neff,ci_neff,hedges_g_neff,hedges_g_ci_neff
19,AA001,Unknown,7.0,14.0,14.310045,-0.882897,3.918827e-01,"(-0.19802544721988924, 0.08236758475460249)",-0.368469,"(-1.2465640193903762, 0.5096263423401565)",5.186586,5.288984,8.269692,-0.641861,5.383447e-01,"(-0.2644164396430612, 0.14875857717777446)",-0.359775,"(-1.471251694108262, 0.751702444961889)"
20,AA002,Unknown,15.0,45.0,46.087408,5.968331,3.209600e-07,"(0.17013950396614902, 0.34329193313459544)",1.312888,"(0.6901276370356034, 1.9356475207034687)",5.188948,10.412285,13.417673,3.117546,7.901220e-03,"(0.07938045344406439, 0.43405098365668005)",1.293359,"(0.20062374165535624, 2.38609386010115)"
21,AA004,Unknown,8.0,64.0,10.185364,-1.061763,3.128702e-01,"(-0.14237606329612412, 0.050323444797645125)",-0.322214,"(-1.0511990806964533, 0.40677199890366955)",4.100174,16.639354,5.982512,-0.703325,5.082863e-01,"(-0.20626828733477182, 0.1142156688362928)",-0.316881,"(-1.3581548649215422, 0.7243924635012702)"
0,B001,Responder,47.0,164.0,68.368988,12.889276,5.710927e-20,"(0.26324901327064854, 0.35967893564182896)",2.264151,"(1.8754769857740463, 2.6528258144579855)",8.525420,65.367450,9.156947,5.790327,2.458726e-04,"(0.19009881973086662, 0.43282912918161087)",2.280736,"(1.4845204263601097, 3.076950991780871)"
12,B002,Non-Responder,5.0,23.0,7.468648,2.176429,6.358087e-02,"(-0.01627825465332633, 0.46347599225693326)",0.865360,"(-0.100554382171541, 1.8312738093488274)",2.250573,8.562263,2.591810,1.419528,2.641242e-01,"(-0.32545057653207277, 0.7726483141356797)",0.810998,"(-0.5715013582784642, 2.1934966647803016)"
1,B004,Responder,NaN,NaN,NaN,NaN,NaN,"(nan, nan)",NaN,"(nan, nan)",NaN,NaN,NaN,NaN,NaN,"(nan, nan)",NaN,"(nan, nan)"
2,B005,Responder,43.0,385.0,54.359228,10.981848,2.048922e-15,"(0.3201680408695482, 0.46315159982362275)",1.615693,"(1.2830082982219757, 1.9483786308683624)",23.861302,54.737910,48.318377,7.030501,6.385576e-09,"(0.279669108674372, 0.503650532018799)",1.635586,"(1.095194520884037, 2.1759782729306294)"
13,B006,Non-Responder,13.0,879.0,32.274289,41.948784,9.931488e-30,"(0.480844325034911, 0.5299076890518192)",2.273096,"(1.71588170945861, 2.8303101975485254)",6.521366,39.411895,43.927163,13.296030,5.314742e-17,"(0.4287692207014769, 0.5819827933852532)",2.371549,"(1.4237426549638486, 3.3193551622582973)"
3,B007,Responder,13.0,62.0,72.738781,4.475020,2.768282e-05,"(0.090051159602381, 0.2346842340433095)",0.679042,"(0.07743352137687365, 1.2806495293752942)",6.244326,9.931591,10.248288,1.914024,8.393137e-02,"(-0.02602768953977999, 0.35076308318547045)",0.740740,"(-0.2401349314564879, 1.7216149906708438)"
14,B008,Non-Responder,18.0,328.0,19.894950,0.740841,4.674446e-01,"(-0.06925736626912245, 0.1455058172747123)",0.149503,"(-0.32406708431653364, 0.6230737157805694)",11.009541,120.680166,12.904006,0.565378,5.815146e-01,"(-0.10766280331704647, 0.18391125432263633)",0.149662,"(-0.46408036810767417, 0.7634045007313273)"


# Pooled Comparison Stats

In [5]:
feature = 'lfp_left_OvER_interpolate_R2'

pooled_results_dict = {'response_status': []}
for corr in ['raw', 'neff']:
    for m in ['n1', 'n2', 'dof', 't_stat', 'p_val', 'ci', 'hedges_g', 'hedges_g_ci']:
        pooled_results_dict[f'{m}_{corr}'] = []

for i, response_status in enumerate(['Responder', 'Non-Responder', 'Unknown']):
    table_df = df.query('response_status == @response_status')
    table_df_daily = table_df.dropna(subset=[feature]).groupby(['pt_id', 'days_since_dbs']).first().reset_index()
    predbs_vals = table_df_daily.query('state_label_str == "Pre-DBS"')[feature]
    predbs_weights = []
    neff_predbs_total = 0.0
    for pt_id, pt_df in table_df_daily.groupby('pt_id'):
        pt_predbs = pt_df.query('state_label_str == "Pre-DBS"')
        num_predbs = pt_predbs[feature].shape[0]
        if num_predbs > 0:
            predbs_weights.extend([1 / num_predbs] * num_predbs)
            if num_predbs > 1:
                neff_predbs_total += compute_neff(pt_predbs['days_since_dbs'].values, pt_predbs[feature].values)
            else:
                neff_predbs_total += num_predbs
    postdbs_vals = table_df_daily.query('state_label_str == @response_status')[feature]
    postdbs_weights = []
    neff_postdbs_total = 0.0
    for pt_id, pt_df in table_df_daily.groupby('pt_id'):
        pt_postdbs = pt_df.query('state_label_str == @response_status')
        num_postdbs = pt_postdbs[feature].shape[0]
        if num_postdbs > 0:
            postdbs_weights.extend([1 / num_postdbs] * num_postdbs)
            if num_postdbs > 1:
                neff_postdbs_total += compute_neff(pt_postdbs['days_since_dbs'].values, pt_postdbs[feature].values)
            else:
                neff_postdbs_total += num_postdbs

    # statistical tests
    if len(predbs_vals) > 1 and len(postdbs_vals) > 1 and not predbs_vals.isna().all() and not postdbs_vals.isna().all():
        n1_raw, n2_raw = len(predbs_vals), len(postdbs_vals)
        raw_stats = welch_stats_with_effect_size(predbs_vals.values, postdbs_vals.values, n1_raw, n2_raw)
        if neff_predbs_total > 1 and neff_postdbs_total > 1:
            neff_stats = welch_stats_with_effect_size(predbs_vals.values, postdbs_vals.values, neff_predbs_total, neff_postdbs_total)
        else:
            neff_stats = dict(_NAN_STATS)
    else:
        raw_stats = dict(_NAN_STATS)
        neff_stats = dict(_NAN_STATS)

    pooled_results_dict['response_status'].append(response_status)
    for corr, stat_dict in [('raw', raw_stats), ('neff', neff_stats)]:
        for m in ['n1', 'n2', 'dof', 't_stat', 'p_val', 'ci', 'hedges_g', 'hedges_g_ci']:
            pooled_results_dict[f'{m}_{corr}'].append(stat_dict[m])

pooled_df = pd.DataFrame(pooled_results_dict)
pooled_df.to_excel('tables/pooled_stats.xlsx', index=False)
pooled_df

,response_status,n1_raw,n2_raw,dof_raw,t_stat_raw,p_val_raw,ci_raw,hedges_g_raw,hedges_g_ci_raw,n1_neff,n2_neff,dof_neff,t_stat_neff,p_val_neff,ci_neff,hedges_g_neff,hedges_g_ci_neff
0,Responder,183,2096,210.186864,20.775980,7.491892e-53,"(0.27988762534862344, 0.3385695071155934)",1.707183,"(1.5482309276475126, 1.8661340723746875)",61.281208,383.124477,77.731559,11.688536,8.478940e-19,"(0.2565564205636958, 0.36190071190052103)",1.697060,"(1.4056606816786337, 1.9884588027977506)"
1,Non-Responder,115,1641,131.902029,-1.828591,6.972010e-02,"(-0.08751144562200319, 0.0034373185965375017)",-0.170333,"(-0.3593993622481788, 0.01873333353028736)",38.941446,240.543006,52.153157,-1.018087,3.133362e-01,"(-0.1248861808082244, 0.04081205378275872)",-0.170414,"(-0.5083453589527739, 0.16751638033330063)"
2,Unknown,30,327,36.302438,4.811175,2.625573e-05,"(0.10263242607605093, 0.25213881870077637)",0.820287,"(0.44236342034279297, 1.1982097006603265)",14.038052,67.815909,20.726545,3.090345,5.603856e-03,"(0.05791994320547719, 0.2968513015713501)",0.822272,"(0.23920886797020313, 1.4053352973091977)"


In [ ]:
# Right Hemisphere
feature = 'lfp_right_OvER_interpolate_R2'

pooled_results_dict = {'response_status': []}
for corr in ['raw', 'neff']:
    for m in ['n1', 'n2', 'dof', 't_stat', 'p_val', 'ci', 'hedges_g', 'hedges_g_ci']:
        pooled_results_dict[f'{m}_{corr}'] = []

for i, response_status in enumerate(['Responder', 'Non-Responder', 'Unknown']):
    table_df = df.query('response_status == @response_status')
    table_df_daily = table_df.dropna(subset=[feature]).groupby(['pt_id', 'days_since_dbs']).first().reset_index()
    predbs_vals = table_df_daily.query('state_label_str == "Pre-DBS"')[feature]
    predbs_weights = []
    neff_predbs_total = 0.0
    for pt_id, pt_df in table_df_daily.groupby('pt_id'):
        pt_predbs = pt_df.query('state_label_str == "Pre-DBS"')
        num_predbs = pt_predbs[feature].shape[0]
        if num_predbs > 0:
            predbs_weights.extend([1 / num_predbs] * num_predbs)
            if num_predbs > 1:
                neff_predbs_total += compute_neff(pt_predbs['days_since_dbs'].values, pt_predbs[feature].values)
            else:
                neff_predbs_total += num_predbs
    postdbs_vals = table_df_daily.query('state_label_str == @response_status')[feature]
    postdbs_weights = []
    neff_postdbs_total = 0.0
    for pt_id, pt_df in table_df_daily.groupby('pt_id'):
        pt_postdbs = pt_df.query('state_label_str == @response_status')
        num_postdbs = pt_postdbs[feature].shape[0]
        if num_postdbs > 0:
            postdbs_weights.extend([1 / num_postdbs] * num_postdbs)
            if num_postdbs > 1:
                neff_postdbs_total += compute_neff(pt_postdbs['days_since_dbs'].values, pt_postdbs[feature].values)
            else:
                neff_postdbs_total += num_postdbs

    # statistical tests
    if len(predbs_vals) > 1 and len(postdbs_vals) > 1 and not predbs_vals.isna().all() and not postdbs_vals.isna().all():
        n1_raw, n2_raw = len(predbs_vals), len(postdbs_vals)
        raw_stats = welch_stats_with_effect_size(predbs_vals.values, postdbs_vals.values, n1_raw, n2_raw)
        if neff_predbs_total > 1 and neff_postdbs_total > 1:
            neff_stats = welch_stats_with_effect_size(predbs_vals.values, postdbs_vals.values, neff_predbs_total, neff_postdbs_total)
        else:
            neff_stats = dict(_NAN_STATS)
    else:
        raw_stats = dict(_NAN_STATS)
        neff_stats = dict(_NAN_STATS)

    pooled_results_dict['response_status'].append(response_status)
    for corr, stat_dict in [('raw', raw_stats), ('neff', neff_stats)]:
        for m in ['n1', 'n2', 'dof', 't_stat', 'p_val', 'ci', 'hedges_g', 'hedges_g_ci']:
            pooled_results_dict[f'{m}_{corr}'].append(stat_dict[m])

pooled_df = pd.DataFrame(pooled_results_dict)
pooled_df.to_excel('tables/right_pooled_stats.xlsx', index=False)
pooled_df

,response_status,n1_raw,n2_raw,dof_raw,t_stat_raw,p_val_raw,ci_raw,hedges_g_raw,hedges_g_ci_raw,n1_neff,n2_neff,dof_neff,t_stat_neff,p_val_neff,ci_neff,hedges_g_neff,hedges_g_ci_neff
0,Responder,183,1320,232.639513,16.452932,7.304257e-41,"(0.25117936042804745, 0.3195197056522505)",1.321656,"(1.1600699195070163, 1.48324261474477)",69.046271,324.927605,97.516202,9.806281,3.340193e-16,"(0.22760063148477416, 0.3430984345955238)",1.318049,"(1.0429671115348613, 1.593130820522492)"
1,Non-Responder,115,1718,129.456014,5.428098,2.710296e-07,"(0.08109091053069953, 0.17410642610072188)",0.526602,"(0.33712496246768703, 0.7160781649286132)",42.216071,276.572665,54.327136,3.166072,2.533575e-03,"(0.04680939953229922, 0.20838793709912218)",0.525258,"(0.1996041746108902, 0.8509122469033922)"
2,Unknown,30,123,42.995383,2.945315,5.189586e-03,"(0.033837259316038454, 0.18080764971774563)",0.611733,"(0.20874775923994482, 1.014717750712307)",14.475708,32.340623,25.060373,1.905065,6.830866e-02,"(-0.008688129913209497, 0.22333303894699358)",0.601737,"(-0.019703817735974183, 1.2231773148341314)"


# LinAR-1 Regression Stats Table 

In [7]:
# Calculate delta features based on pre-DBS means
feature_col = f'lfp_right_OvER_interpolate_R2'

pre_dbs_means = (
    df.query('days_since_dbs < 0')
    .groupby('pt_id')[feature_col]
    .mean()
    .rename(f'pre_dbs_mean_{feature_col}')
    .reset_index()
)
df = df.merge(pre_dbs_means, on='pt_id', how='left')
df[f'delta_{feature_col}'] = df[feature_col] - df[f'pre_dbs_mean_{feature_col}']
df.drop(columns=[f'pre_dbs_mean_{feature_col}'], inplace=True)

In [8]:
daily_df = df.groupby(by=['pt_id', 'days_since_dbs']).head(1)

daily_df = generate_delta_avg_features(daily_df, [f'lfp_{hem}_OvER_interpolate_R2'], hem)
daily_df.to_parquet('data/left_right_r2_features.parq')
print('Saved left-right R2 features to data/left_right_r2_features.parq')

daily_df.query('~pt_id.str.contains("AA").values', inplace=True)
daily_df.columns.values

Saved left-right R2 features to data/left_right_r2_features.parq


array(['pt_id', 'time_bin_time', 'days_since_dbs', 'lfp_left_raw',
       'stim_left', 'lfp_right_raw', 'stim_right', 'lead_location',
       'left_lead_model', 'right_lead_model', 'active_left_freq',
       'active_left_pulse_width', 'active_left_suspend_amplitude',
       'active_left_lower_amplitude', 'active_left_upper_amplitude',
       'active_left_stim_cathodes', 'active_left_stim_anodes',
       'active_left_sensing_contacts', 'active_left_sensing_frequency',
       'active_right_freq', 'active_right_pulse_width',
       'active_right_suspend_amplitude', 'active_right_lower_amplitude',
       'active_right_upper_amplitude', 'active_right_stim_cathodes',
       'active_right_stim_anodes', 'active_right_sensing_contacts',
       'active_right_sensing_frequency',
       'lfp_left_OvER_interpolate_z_scored',
       'lfp_right_OvER_interpolate_z_scored', 'contig_right',
       'contig_left', 'lfp_left_OvER_interpolate_z_scored_lag_1',
       'lfp_right_OvER_interpolate_z_scored_lag_

In [9]:
reg_results = pd.DataFrame(columns = ['Hemisphere', 'Feature', 'Label', 'AUROC', 'BA', 'TPR', 'TNR'])

for Hemisphere in ['left', 'right']:
    for Feature in ['daily', 'delta', 'avg']:
        match Feature:
            case 'daily':
                feature = [f'lfp_{Hemisphere}_OvER_interpolate_R2']
            case 'delta':
                feature = [f'delta_lfp_{Hemisphere}_OvER_interpolate_R2']
            case 'avg':
                feature = [f'{Hemisphere}_delta_R2_rolling_avg_14d']

        results_dict = reg(daily_df, feature)

        reg_results.loc[len(reg_results)] = [Hemisphere, Feature, 'true', results_dict[0]['AUC'], results_dict[0]['balanced_accuracy'], results_dict[0]['true_positive_rate'], results_dict[0]['true_negative_rate']]
        
        AUROC_dist = []
        BA_dist = []
        TPR_dist = []
        TNR_dist = []

        def run_shuffle(_):
            shuffle_results = reg(daily_df, feature, shuffle=True)
            return shuffle_results[0]['AUC'], shuffle_results[0]['balanced_accuracy'], shuffle_results[0]['true_positive_rate'], shuffle_results[0]['true_negative_rate']

        n = 10000
        n_jobs = -1  # uses all available cores
        results = Parallel(n_jobs=n_jobs)(
            delayed(run_shuffle)(i) for i in tqdm(range(n), desc=f'Shuffling labels for {Feature} model, {Hemisphere} hem')
        )

        AUROC_dist, BA_dist, TPR_dist, TNR_dist = zip(*results)

        reg_results.loc[len(reg_results)] = [Hemisphere, Feature, 'shuffled', np.mean(AUROC_dist), np.mean(BA_dist), np.mean(TPR_dist), np.mean(TNR_dist)]

        def calc_p(dist, metric):
            return (np.sum(np.array(dist) >= reg_results[(reg_results.Hemisphere == Hemisphere) & (reg_results.Feature == Feature) & (reg_results.Label == 'true')][metric].values[0]) + 1) / (len(dist) + 1)
        
        AUROC_p = calc_p(AUROC_dist, 'AUROC')
        BA_p = calc_p(BA_dist, 'BA')
        TPR_p = calc_p(TPR_dist, 'TPR')
        TNR_p = calc_p(TNR_dist, 'TNR')
        reg_results.loc[len(reg_results)] = [Hemisphere, Feature, 'p_val', AUROC_p, BA_p, TPR_p, TNR_p]

reg_results.to_excel('tables/all_reg_stats.xlsx')

Shuffling labels for avg model, right hem: 100%|██████████| 10000/10000 [13:29<00:00, 12.35it/s]


# Per-fold stats

In [10]:
model_features = ['delta_lfp_left_OvER_interpolate_R2']
daily_df = df.dropna(subset=model_features, how='any').groupby(['pt_id', 'days_since_dbs']).head(1)

results, pt_results, overall_model = leave_one_patient_out_logistic_regression(
    daily_df,
    model_features,
)
perfold_dict = {
    'pt_id': [],
    'response_status': [],
    'raw_acc': [],
    'decision_threshold': []
}
for pt_id, this_pt_results in pt_results.items():
    response_status = daily_df.query('pt_id == @pt_id')['response_status'].iloc[0]
    try:
        raw_acc = this_pt_results['raw_accuracy']
    except KeyError:
        raw_acc = np.nan
    try:
        w0 = this_pt_results['model'].named_steps['logreg'].intercept_[0]
        w1 = this_pt_results['model'].named_steps['logreg'].coef_[0][0]
        decision_threshold = -w0 / w1
    except KeyError:
        decision_threshold = np.nan

    perfold_dict['pt_id'].append(pt_id)
    perfold_dict['response_status'].append(response_status)
    perfold_dict['raw_acc'].append(raw_acc)
    perfold_dict['decision_threshold'].append(decision_threshold)

perfold_df = pd.DataFrame(perfold_dict)
perfold_df.to_excel('tables/perfold_logreg_stats.xlsx', index=False)
perfold_df

,pt_id,response_status,raw_acc,decision_threshold
0,AA001,Unknown,NaN,-0.185306
1,AA002,Unknown,NaN,-0.185423
2,AA004,Unknown,NaN,-0.185374
3,B001,Responder,0.959259,-0.172971
4,B002,Non-Responder,0.916667,-0.185299
5,B004,Responder,0.950311,-0.173257
6,B005,Responder,0.584416,-0.204570
7,B006,Non-Responder,0.903431,-0.187650
8,B007,Responder,NaN,-0.185343
9,B008,Non-Responder,0.726496,-0.178981


# Regression Stats for LinAR-1, LinAR-k, and LinAR-k zoned models including old and new data sets

In [11]:
# Load LinAR-k and LinAR-k zoned data
ark_df = pd.read_parquet('data/df_w_ark.parq')
arkz_df = pd.read_pickle('data/ark_zoned_all_pts_r2.pkl')

# Compile delta and averaged features
reg_df = pd.merge(daily_df, ark_df[['pt_id', 'days_since_dbs', 'lfp_left_OvER_interpolate_R2_ark']].drop_duplicates(subset=['pt_id', 'days_since_dbs']), on=['days_since_dbs', 'pt_id'], how='left', suffixes=(None, None))
reg_df = pd.merge(reg_df, arkz_df, on=['days_since_dbs', 'pt_id'], how='left', suffixes=(None, '_arkz'))
reg_df.rename(columns={'r2': 'R2_arkz'}, inplace=True)

# Generate features for old data set and all data
OLD_PAPER_PTS = {'B001': (-48, 100), 'B002': (-6, 296), 'B004': (-9, 803), 'B005': (-44, 578), 'B006': (-13, 585), 'B007': (-13, -1), 'B008': (-19, 141), 'B009': (1106, 1224), 'B010': (-51, -29), 'U001': (-25, 43), 'U003': (-20, 14)}
old_data = []
for pt in list(OLD_PAPER_PTS.keys()):
    (start, end) = OLD_PAPER_PTS[pt]
    old_data.append(reg_df.query('pt_id == @pt and @start <= days_since_dbs <= @end'))

old_data_df = pd.concat(old_data)
old_data_df = generate_delta_avg_features(old_data_df, ['lfp_left_OvER_interpolate_R2', 'lfp_left_OvER_interpolate_R2_ark', 'R2_arkz'], 'left')

reg_df = generate_delta_avg_features(reg_df, ['lfp_left_OvER_interpolate_R2_ark', 'R2_arkz'], 'left')
reg_df.head(5)

,pt_id,time_bin_time,days_since_dbs,lfp_left_raw,stim_left,lfp_right_raw,stim_right,lead_location,left_lead_model,right_lead_model,...,left_arkz_rolling_avg_14d,left_arkz_rolling_avg_21d,left_arkz_rolling_avg_28d,left_delta_arkz_rolling_avg_1d,left_delta_arkz_rolling_avg_3d,left_delta_arkz_rolling_avg_5d,left_delta_arkz_rolling_avg_7d,left_delta_arkz_rolling_avg_14d,left_delta_arkz_rolling_avg_21d,left_delta_arkz_rolling_avg_28d
0,AA001,00:00:00,-6,113.0,0.0,373.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AA001,00:00:00,-5,275.0,0.0,494.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AA001,00:00:00,-4,211.0,0.0,358.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AA001,00:00:00,-3,394.0,0.0,599.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AA001,00:00:00,-2,392.0,0.0,573.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# Regression stats AR(1), AR(k), and AR(k)-zoned models for both old and new data

reg_results = pd.DataFrame(columns = ['Model', 'Feature', 'Data', 'AUROC', 'BA', 'TPR', 'TNR'])
model_map = {'LinAR-1': 'lfp_left_OvER_interpolate_R2', 'LinAR-k': 'lfp_left_OvER_interpolate_R2_ark', 'LinAR-k (zoned)': 'R2_arkz'}

for Model in ['LinAR-1', 'LinAR-k', 'LinAR-k (zoned)']: # AR-1, AR-k, AR-k zoned
    for Feature in ['daily', 'delta', 'avg']:
        match Feature:
            case 'daily':
                feature = [model_map[Model]]
            case 'delta':
                feature = [f'delta_{model_map[Model]}']
            case 'avg':
                feature = [f'left_delta_{model_map[Model].split('_')[-1]}_rolling_avg_14d']
        for Data in ['old', 'all']:
            fold_df = old_data_df if Data == 'old' else reg_df
            print(f'Running regression for {Model} with {Feature.upper()} feature for {Data.upper()} data...')
            _results = reg(fold_df, feature)

            reg_results.loc[len(reg_results)] = [Model, Feature.upper(), Data.upper(), _results[0]['AUC'], _results[0]['balanced_accuracy'], _results[0]['true_positive_rate'], _results[0]['true_negative_rate']]

reg_results.to_excel('tables/ar_reg_stats.xlsx')

Running regression for LinAR-1 with DAILY feature for OLD data...
Running regression for LinAR-1 with DAILY feature for ALL data...
Running regression for LinAR-1 with DELTA feature for OLD data...
Running regression for LinAR-1 with DELTA feature for ALL data...
Running regression for LinAR-1 with AVG feature for OLD data...
Running regression for LinAR-1 with AVG feature for ALL data...
Running regression for LinAR-k with DAILY feature for OLD data...
Running regression for LinAR-k with DAILY feature for ALL data...
Running regression for LinAR-k with DELTA feature for OLD data...
Running regression for LinAR-k with DELTA feature for ALL data...
Running regression for LinAR-k with AVG feature for OLD data...
Running regression for LinAR-k with AVG feature for ALL data...
Running regression for LinAR-k (zoned) with DAILY feature for OLD data...
Running regression for LinAR-k (zoned) with DAILY feature for ALL data...
Running regression for LinAR-k (zoned) with DELTA feature for OLD da

# Delong's test to compare model performances

In [13]:
# Delong's results for AR(k) vs AR(k)-zoned and AR(k) vs AR(1) models for both old and new data
def concat_dict(dict, key):
    arr = []
    for pt in list(dict.keys()):
        try:
            arr.extend(dict[pt][key])
        except KeyError:
            continue
    return np.array(arr)

delong_results = pd.DataFrame(columns=['Feature', 'Other Model', 'p', 'z', 'LinAR-1 AUC', 'Other Model AUC'])
for Feature in ['daily', 'delta', 'avg']:

    for Model in ['LinAR-k', 'LinAR-k (zoned)']:
            match Feature:
                case 'daily':
                    ar1_feature = [model_map['LinAR-1']]
                    feature = [model_map[Model]]
                case 'delta':
                    ar1_feature = [f'delta_{model_map['LinAR-1']}']
                    feature = [f'delta_{model_map[Model]}']
                case 'avg':
                    ar1_feature = [f'left_delta_{model_map['LinAR-1'].split('_')[-1]}_rolling_avg_14d']
                    feature = [f'left_delta_{model_map[Model].split('_')[-1]}_rolling_avg_14d']

            print(f'Comparing regressions for {Model} and LinAR-1 with {Feature} feature ...')

            fold_df = reg_df.dropna(subset=[*ar1_feature, *feature])
            ar1_results = reg(fold_df, ar1_feature)
            y_true = concat_dict(ar1_results[1], 'y_true')
            ar1_probs = concat_dict(ar1_results[1], 'y_prob')

            ark_results = reg(fold_df, feature)
            ark_probs = concat_dict(ark_results[1], 'y_prob')
        
            z, p, auc_ar1, auc_ark = Delong_test(y_true, ar1_probs, ark_probs, return_ci=False, return_auc=True, verbose=0)

            delong_results.loc[len(delong_results)] = [Feature, Model, p, z, auc_ar1, auc_ark]

delong_results['AUC Diff'] = delong_results['LinAR-1 AUC'] - delong_results['Other Model AUC']
delong_results.to_excel('tables/ark_delong_test_results.xlsx')

Comparing regressions for LinAR-k and LinAR-1 with daily feature ...
Comparing regressions for LinAR-k (zoned) and LinAR-1 with daily feature ...
Comparing regressions for LinAR-k and LinAR-1 with delta feature ...
Comparing regressions for LinAR-k (zoned) and LinAR-1 with delta feature ...
Comparing regressions for LinAR-k and LinAR-1 with avg feature ...
Comparing regressions for LinAR-k (zoned) and LinAR-1 with avg feature ...
